# Calibration 3 — Nhân quả trong đuôi theo thời gian (Granger trong đuôi)

EASE chỉ dùng quan hệ **đồng thời**. Notebook này hỏi câu hỏi **có thời gian**: cú sốc cực trị của mã $X$ hôm nay có làm mã $Y$ dễ cực trị trong $p$ phiên tới
hơn mức thông thường không — sau khi loại các phiên mà chính $Y$ hoặc thị trường đã cực trị?

**Hệ số** (theo Bodik & Pasche 2024, "Granger causality in extremes", arXiv:2407.09632; bản đơn giản hoá — điều kiện chỉ xét phiên $t$):
$$\hat\Gamma_{X\to Y}(p) = \frac{1}{|S|}\sum_{t\in S}\max_{1\le h\le p}\tilde F_Y(Y_{t+h}),\qquad \tilde F_Y(y) = \hat F_Y(y)\,\mathbb 1\{\hat F_Y(y)\ge \tfrac12\},$$
$S$: $k$ phiên có $X_t$ cực trị trong số các phiên "sạch" ($\hat F_Y(Y_t) \le 0{,}8$ và thị trường không chứa $X, Y$ có $\hat F_M(M_t) \le 0{,}9$).
Mức nền $b$ = cùng đại lượng trên mọi phiên sạch; $\Delta = \hat\Gamma - b$. Kiểm định $H_0: \Delta \le 0$ bằng **moving-block bootstrap** (khối $\lceil\sqrt n\rceil$),
hiệu chỉnh **Benjamini–Hochberg** trên 380 cặp. Đuôi dưới (cú sập) là trọng tâm; đuôi trên để so sánh.

## 0. Thiết lập

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import eda_utils as U
import causal_tail as CT
import causal_baselines as CB
import tail_preprocessing as TP

U.apply_publication_style()
pd.set_option("display.float_format", "{:,.3f}".format)
SYMBOLS, COL, PAL = U.SYMBOLS, U.SYMBOL_COLORS, U.PALETTE
FOLDER = "calibration"                      # hình → report/figure/processing, bảng → report/table/processing
METHOD_COLORS = {"EASE": PAL["blue_main"], "Pairwise LiNGAM": PAL["red_strong"], "ICA-LiNGAM": PAL["green_3"],
                 "Rank PC": PAL["violet"], "Ngẫu nhiên": PAL["neutral"]}

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import validation as V
import calibration as C
from scipy import stats
from joblib import Parallel, delayed
P = len(SYMBOLS)
OFF = ~np.eye(P, dtype=bool)
Z = pd.read_parquet(ROOT / "data" / "processed" / "garch_std_resid_market.parquet")
z = Z[SYMBOLS]
n, K = len(z), CT.default_k(len(z))
NU_CAL = 0.35                     # ν chọn ở Calibration 01 (mô phỏng hiệu chỉnh + độ ổn định)
KC = CT.default_k(n, NU_CAL)
print(f"n = {n} phiên, {P} mã, k(ν=0,4) = {K}, k hiệu chỉnh (ν={NU_CAL}) = {KC}")

SECTOR = U.SECTOR
B_BOOT = 300

## 1. Kiểm tra kích thước kiểm định (size) dưới $H_0$

**Các bước:** với mỗi độ trễ $p \in \{1, 3, 5\}$: 3 bộ dữ liệu từ mô hình một nhân tố **không có độ trễ** (không có nhân quả theo thời gian), cùng $n$,
chạy đủ 380 cặp; tỷ lệ $p < 0{,}05$ phải ≈ 5% và số cặp qua FDR 10% phải ≈ 0. $k = \lfloor n^{0{,}35}\rfloor = 14$ (Calibration 01; gần mặc định $n^{1/3}$ của bài báo).

In [ ]:
gp = pd.read_csv(ROOT / "report" / "table" / "processing" / "02_garch_params.csv", index_col=0, encoding="utf-8-sig")
NU = float(gp["nu"].mean()); lam = V.fit_one_factor(z).to_numpy()
size = []
for lag in (1, 3, 5):
    for rep in range(3):
        X = pd.DataFrame(V.simulate_factor_scm(n, lam, NU, rng=np.random.default_rng(500 + rep)), columns=SYMBOLS)
        G0 = C.extremal_granger_matrix(X, k=KC, lag=lag, tail="lower", B=B_BOOT, seed=rep)
        size.append({"tre": lag, "lan": rep, "ty_le_p<0,05": (G0.p < 0.05).mean(), "so_cap_q<0,1": int((G0.q < 0.1).sum())})
size = pd.DataFrame(size).set_index(["tre", "lan"])
U.save_table(size, "03_size_check", FOLDER)
size

### Phân tích

_(điền sau khi chạy)_

## 2. Dữ liệu thật: đuôi dưới, độ trễ 1, 3, 5 phiên

**Tính gì:** $\hat\Gamma$, $b$, $\Delta$, p, q cho 380 cặp với $p \in \{1, 3, 5\}$ (đuôi dưới) và $p = 1$ (đuôi trên). Hình: ma trận $\Delta$ (đuôi dưới, $p=1$)
và phân phối p-value cho từng cấu hình.

In [ ]:
configs = [("lower", 1), ("lower", 3), ("lower", 5), ("upper", 1)]
res = {}
for tail, lag in configs:
    G = C.extremal_granger_matrix(z, k=KC, lag=lag, tail=tail, B=B_BOOT, seed=lag * 10 + (tail == "upper"))
    res[(tail, lag)] = G
    U.save_table(G.set_index(["cause", "effect"]), f"03_granger_{tail}_lag{lag}", FOLDER)
summary = pd.DataFrame({f"{t}, p={l}": {"ty_le_p<0,05": (G.p < 0.05).mean(), "so_cap_q<0,1": int((G.q < 0.1).sum()),
                                        "Delta_TB": G.delta.mean(), "so_cap_quy_tac_cuc_tri": int(G.extreme_rule.sum())}
                        for (t, l), G in res.items()}).T
U.save_table(summary, "03_granger_summary", FOLDER)

G1 = res[("lower", 1)]
Dm = G1.pivot(index="cause", columns="effect", values="delta").reindex(index=SYMBOLS, columns=SYMBOLS)
fig, axes = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw={"width_ratios": [1.25, 1]})
U.make_heatmap(axes[0], Dm, cmap="RdBu_r", vmin=-0.3, vmax=0.3, fmt=None, tick_fontsize=9, cbar_label=r"$\Delta = \hat\Gamma - b$",
               title="Đuôi dưới, trễ 1 phiên (hàng → cột)")
axes[0].set_xlabel("Hệ quả (phiên sau)"); axes[0].set_ylabel("Nguyên nhân (hôm nay)")
for (t, l), G in res.items():
    axes[1].hist(G.p, bins=np.linspace(0, 1, 21), histtype="step", lw=1.8, label=f"{t}, p={l}")
axes[1].axhline(380 / 20, color=PAL["gray_dark"], ls="--", lw=1, label="kỳ vọng dưới H0")
axes[1].set_xlabel("p-value"); axes[1].set_ylabel("Số cặp"); axes[1].legend(fontsize=9)
U.finalize_figure(fig, "03_granger_real", folder=FOLDER); plt.show()
summary.round(3)

In [ ]:
# Tổng hợp theo ngành (đuôi dưới, trễ 1): Δ trung bình từ ngành nguồn tới ngành đích
G1s = G1.assign(nganh_nguon=G1.cause.map(SECTOR), nganh_dich=G1.effect.map(SECTOR))
blk = G1s.groupby(["nganh_nguon", "nganh_dich"]).delta.mean().unstack().reindex(index=U.SECTORS, columns=U.SECTORS)
U.save_table(blk, "03_granger_sector_blocks", FOLDER)
print("Cặp mạnh nhất (đuôi dưới, trễ 1):")
G1.sort_values("p").head(10)[["cause", "effect", "gamma", "baseline", "delta", "p", "q"]].round(3)

### Phân tích

_(điền sau khi chạy)_

## 3. Kết luận

_(điền sau khi chạy)_